# 📦 Amazon Parcel Defect Detection with YOLOv8-OBB (Colab Edition with Training + Pretrained Inference)
This enhanced notebook:
- Clones GitHub dataset
- Shows training image samples
- Trains for 1 epoch to demonstrate learning
- Then loads a pretrained checkpoint for evaluation and predictions
- Visualizes predictions on test images

In [1]:
# --- 🔧 Step 1: Install YOLOv8 ---
!pip install ultralytics -q

In [ ]:
# --- 📥 Step 2: Clone the GitHub Dataset ---
!git clone https://github.com/saikisri97/Optimisation.git

In [2]:
# --- 🛠️ Step 3: Patch data.yaml for Colab paths ---
import os

yaml_path = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml"
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

with open(yaml_path, "w") as f:
    f.write("""train: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
val: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
test: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
nc: 2
names: ['defect parcel', 'no defect parcel']
""")

OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
# --- 🖼️ Step 4: Visualize Sample Training Images ---
import glob
import matplotlib.pyplot as plt
from PIL import Image

image_dir = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images"
image_files = sorted(glob.glob(f"{image_dir}/*.jpg") + glob.glob(f"{image_dir}/*.png"))

plt.figure(figsize=(12, 6))
for i, image_file in enumerate(image_files[:4]):
    img = Image.open(image_file)
    plt.subplot(1, 4, i+1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Sample {i+1}")
plt.suptitle("🔍 Example Training Images")
plt.show()


In [ ]:
# --- 🚆 Step 5: Train YOLOv8-OBB for 1 Epoch ---
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")

model.train(
    data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    name="parcel_defect_yolo_demo",
    workers=2
)

In [ ]:
# --- 🧠 Step 6: Load Pretrained Checkpoint from GitHub ---
# (Used for better test set inference after demo training)

pretrained_url = "https://github.com/saikisri97/Optimisation/raw/master/For_AI_Lecture/data/yolov8n-obb-parcel-defect.pt"
checkpoint_path = "/content/yolov8n-obb-parcel-defect.pt"

!wget -O {checkpoint_path} {pretrained_url}

# Load checkpoint
model = YOLO(checkpoint_path)


In [ ]:
# --- 📊 Step 7: Evaluate Pretrained Model on Test Set ---
metrics = model.val(data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml", split='test')
print("Evaluation Metrics:", metrics)

In [ ]:
# --- 🔍 Step 8: Show Predictions from Pretrained Model ---
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

results = model.predict(
    source="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images",
    save=True,
    imgsz=640
)

output_dir = Path(results[0].save_dir)
output_images = list(output_dir.glob("*.jpg"))

plt.figure(figsize=(12, 8))
for i, img_path in enumerate(output_images[:4]):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.title(f"Prediction {i+1}")
    plt.axis("off")
plt.suptitle("📦 YOLOv8 Inference with Pretrained Checkpoint")
plt.tight_layout()
plt.show()
